# Improves version 3

In [3]:
!pip -q install statsmodels scikit-learn joblib

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from joblib import dump

warnings.filterwarnings("ignore")
np.random.seed(42)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)



# LOAD and CLEAN


DATA_PATH = "/content/Final_MasterDataset_upt.csv"
# DATA_PATH = "/mnt/data/Final_MasterDataset_upt.csv"

df = pd.read_csv(DATA_PATH)

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\n", "", regex=False)
        .str.strip()
    )
    rename_map = {
        "PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)": "PM25",
        "Environmental_impact(CO2e/capita)": "CO2e_per_capita",
        "Secondary_SclEnroll_Gross _%": "Secondary_SclEnroll_Gross_Pct",
        "EduExp_GovShare_IMF_%": "EduExp_GovShare_IMF_Pct",
        "Total number of deaths": "Total_Deaths",
        "Asylum Seekers": "Asylum_Seekers",
    }
    return df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

df = clean_columns(df)
df["Year"] = df["Year"].astype(int)
df = df.sort_values(["Country", "Year"]).reset_index(drop=True)

# Drop all-NaN columns
all_na_cols = [c for c in df.columns if df[c].isna().all()]
if all_na_cols:
    print("Dropping all-NaN columns:", all_na_cols)
    df = df.drop(columns=all_na_cols)

print("Shape:", df.shape)
print("Countries:", df["Country"].unique())
print("Year range:", df["Year"].min(), "-", df["Year"].max())



# CONFIG (Education)

TARGET = "Education_Expenditure_GDP"

TRAIN_START, TRAIN_END = 1994, 2016
TEST_START, TEST_END   = 2017, 2022

COUNTRIES = ["USA", "RUS", "CHN", "GBR", "FRA"]

OUTPUT_DIR = Path("education_bestof_outputs_v2")
OUTPUT_DIR.mkdir(exist_ok=True)



# 3) FEATURE ENGINEERING (FIXED)


df["D_PANDEMIC"] = (df["Year"] >= 2020).astype(int)

# Optional extra step dummies
# df["D_POST_2008"] = (df["Year"] >= 2008).astype(int)
# df["D_POST_2010"] = (df["Year"] >= 2010).astype(int)

BASE_X = [
    "D_Expenditure_GDP",
    "GDP_Growth_Annual",
    "Inflation_Annual",
    "Unemployment_Total_of_TLF",
    "Population",
]
EDU_EXTRAS = [
    "EduExp_GovShare_IMF_Pct",
    "Secondary_SclEnroll_Gross_Pct",
]

BASE_X = [c for c in BASE_X if c in df.columns]
EDU_EXTRAS = [c for c in EDU_EXTRAS if c in df.columns]

CONT_FEATS = list(dict.fromkeys(BASE_X + EDU_EXTRAS))

# Dummies
DUMMY_FEATS = [c for c in ["D_PANDEMIC"] if c in df.columns]

# Reduce feature explosion: use ONLY lag1 (lag2 + diff often hurts annual data)
for col in CONT_FEATS:
    df[f"{col}_lag1"] = df.groupby("Country")[col].shift(1)

LAG_FEATS = [f"{c}_lag1" for c in CONT_FEATS if f"{c}_lag1" in df.columns]

# Candidates = levels + lag1 + dummies
X_COLS_ALL = [c for c in (CONT_FEATS + LAG_FEATS + DUMMY_FEATS) if c in df.columns]
print("Candidate Education X columns:", X_COLS_ALL)



# 4) HELPERS

def split_train_test(country_df: pd.DataFrame):
    train_mask = (country_df["Year"] >= TRAIN_START) & (country_df["Year"] <= TRAIN_END)
    test_mask  = (country_df["Year"] >= TEST_START) & (country_df["Year"] <= TEST_END)
    return country_df.loc[train_mask].copy(), country_df.loc[test_mask].copy()

def clean_target_train_only(y_train: pd.Series):
    y = y_train.astype(float).copy()
    y = y.interpolate(limit_direction="both").ffill().bfill()
    y = y.clip(lower=0)
    return y

def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(np.abs(y_true) < 1e-8, np.nan, y_true)
    return np.nanmean(np.abs((y_true - y_pred) / denom)) * 100.0

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) if len(y_true) >= 2 else np.nan
    mape = safe_mape(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "R2": r2}

def mase(y_true, y_pred, y_train):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    if len(y_train) < 2:
        return np.nan
    denom = np.mean(np.abs(y_train[1:] - y_train[:-1]))
    if denom < 1e-8:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / denom

def rolling_naive_baseline(y_train_clean: np.ndarray, y_test_true: np.ndarray):
    last = float(y_train_clean[-1])
    preds = []
    for i in range(len(y_test_true)):
        preds.append(last)
        if not np.isnan(y_test_true[i]):
            last = float(y_test_true[i])
    return np.array(preds, dtype=float)

def rolling_drift_baseline(y_train_clean: np.ndarray, y_test_true: np.ndarray):
    y_hist = np.asarray(y_train_clean, dtype=float).copy()
    preds = []
    for i in range(len(y_test_true)):
        n = len(y_hist)
        slope = (y_hist[-1] - y_hist[0]) / max(n - 1, 1) if n >= 2 else 0.0
        yhat = y_hist[-1] + slope
        preds.append(float(yhat))
        if not np.isnan(y_test_true[i]):
            y_hist = np.append(y_hist, float(y_test_true[i]))
        else:
            y_hist = np.append(y_hist, np.nan)
    return np.array(preds, dtype=float)

def rolling_holt_baseline(y_train_clean: np.ndarray, y_test_true: np.ndarray):
    y_hist = np.asarray(y_train_clean, dtype=float).copy()
    preds = []
    for i in range(len(y_test_true)):
        try:
            model = ExponentialSmoothing(y_hist, trend="add", damped_trend=True)
            res = model.fit(optimized=True)
            yhat = float(np.asarray(res.forecast(1))[0])
        except Exception:
            yhat = float(y_hist[-1])
        preds.append(yhat)
        if not np.isnan(y_test_true[i]):
            y_hist = np.append(y_hist, float(y_test_true[i]))
        else:
            y_hist = np.append(y_hist, np.nan)
    return np.array(preds, dtype=float)

def prune_features_by_train(X_train: pd.DataFrame, X_test: pd.DataFrame, always_keep=None, max_missing=0.35):
    """
    Keeps 'always_keep' columns even if constant in training.
    """
    always_keep = set(always_keep or [])
    keep = []
    for c in X_train.columns:
        if c in always_keep:
            keep.append(c)
            continue
        if X_train[c].isna().mean() > max_missing:
            continue
        if X_train[c].dropna().nunique() <= 1:
            continue
        keep.append(c)
    return X_train[keep].copy(), X_test[keep].copy(), keep

def impute_exog_no_leakage(X_train: pd.DataFrame, X_test: pd.DataFrame):
    """
    No leakage:
      TRAIN: interpolate -> ffill -> fill by TRAIN median
      TEST : ffill from last TRAIN row -> fill by TRAIN median
    """
    Xtr = X_train.copy()
    Xte = X_test.copy()

    Xtr = Xtr.interpolate(limit_direction="forward").ffill()
    med = Xtr.median(numeric_only=True)
    Xtr = Xtr.fillna(med)

    if len(Xtr) > 0 and len(Xte) > 0:
        starter = Xtr.iloc[[-1]]
        combo = pd.concat([starter, Xte], axis=0)
        combo = combo.ffill().iloc[1:]
        Xte = combo

    Xte = Xte.fillna(med)
    return Xtr, Xte

def scale_exog_no_leakage(X_train_imp: pd.DataFrame, X_test_imp: pd.DataFrame):
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(X_train_imp.values)
    Xte_s = scaler.transform(X_test_imp.values)
    return Xtr_s, Xte_s, scaler

def transform_y(y: np.ndarray, use_log: bool, eps=1e-6):
    y = np.asarray(y, dtype=float)
    if use_log:
        return np.log(np.clip(y, 0, None) + eps)
    return y

def invert_y(yhat: np.ndarray, use_log: bool, eps=1e-6):
    yhat = np.asarray(yhat, dtype=float)
    if use_log:
        out = np.exp(yhat) - eps
        return np.clip(out, 0, None)
    return yhat



# 5) TIME-SERIES CV FOLDS

def make_folds(years_train, val_window=2, n_folds=3):
    years = np.sort(np.unique(np.asarray(years_train)))
    if len(years) < 14:
        return []
    total_val = val_window * n_folds
    if len(years) < (10 + total_val):
        n_folds = 2
        total_val = val_window * n_folds
        if len(years) < (10 + total_val):
            return []
    folds = []
    end = len(years)
    for f in range(n_folds, 0, -1):
        val_end = end - (f-1)*val_window
        val_start = val_end - val_window
        folds.append((years[val_start], years[val_end-1]))
    return folds



# 6) EXPANDING-REFIT SARIMAX FORECAST (RELAXED CONSTRAINTS)

def expanding_refit_forecast_sarimax(
    y_train_hist: np.ndarray,
    X_train_hist,
    y_future: np.ndarray,
    X_future,
    order: tuple,
    trend: str,
    use_log: bool,
    eps=1e-6
):
    y_hist = np.asarray(y_train_hist, dtype=float).copy()
    y_future = np.asarray(y_future, dtype=float)
    preds = []

    X_hist = None
    if X_train_hist is not None:
        X_hist = np.asarray(X_train_hist, dtype=float).copy()
        X_future = np.asarray(X_future, dtype=float)

    for i in range(len(y_future)):
        y_fit = transform_y(y_hist, use_log, eps=eps)

        model = SARIMAX(
            y_fit,
            exog=X_hist,
            order=order,
            trend=trend,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        res = model.fit(disp=False)

        fc = res.get_forecast(steps=1, exog=None if X_future is None else X_future[i:i+1])
        yhat_t = np.asarray(fc.predicted_mean)[0]
        yhat = invert_y(np.array([yhat_t]), use_log, eps=eps)[0]
        preds.append(float(yhat))

        # append actual y
        if not np.isnan(y_future[i]):
            y_hist = np.append(y_hist, float(max(y_future[i], 0.0)))
        else:
            y_hist = np.append(y_hist, np.nan)

        # append X
        if X_hist is not None:
            X_hist = np.vstack([X_hist, X_future[i:i+1]])

    return np.array(preds, dtype=float)

def cv_rmse_sarimax(
    y_train_clean: np.ndarray,
    X_train_scaled,
    years_train: np.ndarray,
    order: tuple,
    trend: str,
    use_log: bool,
    val_window=2,
    n_folds=3
):
    folds = make_folds(years_train, val_window=val_window, n_folds=n_folds)
    if not folds:
        return np.inf

    years_train = np.asarray(years_train)
    y_all = np.asarray(y_train_clean, dtype=float)

    rmses = []
    for (val_start, val_end) in folds:
        sub_mask = years_train < val_start
        val_mask = (years_train >= val_start) & (years_train <= val_end)

        if sub_mask.sum() < 10 or val_mask.sum() < 2:
            continue

        y_sub = y_all[sub_mask]
        y_val = y_all[val_mask]

        if X_train_scaled is None:
            X_sub = None
            X_val = None
        else:
            X_sub = X_train_scaled[sub_mask]
            X_val = X_train_scaled[val_mask]

        try:
            preds_val = expanding_refit_forecast_sarimax(
                y_train_hist=y_sub,
                X_train_hist=X_sub,
                y_future=y_val,
                X_future=X_val,
                order=order,
                trend=trend,
                use_log=use_log
            )
            mask = ~np.isnan(y_val)
            rmse = np.sqrt(mean_squared_error(y_val[mask], preds_val[mask]))
            rmses.append(float(rmse))
        except Exception:
            rmses.append(np.inf)

    if len(rmses) == 0:
        return np.inf
    return float(np.mean(rmses))

def select_best_sarimax_spec(y_train_clean, X_train_scaled, years_train):
    """
    Keep grid small for yearly data.
    """
    p_list = [0, 1, 2]
    q_list = [0, 1, 2]
    d_list = [0, 1]
    trends = ["n", "c"]
    logs = [False, True]

    best = None
    best_rmse = np.inf

    for use_log in logs:
        for d in d_list:
            for trend in trends:
                for p in p_list:
                    for q in q_list:
                        order = (p, d, q)
                        rmse = cv_rmse_sarimax(
                            y_train_clean=y_train_clean,
                            X_train_scaled=X_train_scaled,
                            years_train=years_train,
                            order=order,
                            trend=trend,
                            use_log=use_log,
                            val_window=2,
                            n_folds=3
                        )
                        if rmse < best_rmse:
                            best_rmse = rmse
                            best = (order, trend, use_log)

    if best is None:
        best = ((0, 1, 0), "c", False)
        best_rmse = np.inf

    return best, best_rmse



# 7) TRAIN + EVAL

def train_evaluate_education_bestof(target: str, x_cols_all: list[str]):
    all_metrics = []
    all_preds = []

    target_dir = OUTPUT_DIR / target
    models_dir = target_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    EXOG_MODES = ["NO_EXOG_ARIMA", "DUMMIES_ONLY", "TOPK_EXOG"]

    for country in COUNTRIES:
        print("\n" + "="*110)
        print("Country:", country)

        cdf = df[df["Country"] == country].copy()
        cdf = cdf[(cdf["Year"] >= TRAIN_START) & (cdf["Year"] <= TEST_END)].sort_values("Year").reset_index(drop=True)

        train_df, test_df = split_train_test(cdf)

        # y
        y_train_clean = clean_target_train_only(train_df[target])
        y_test = test_df[target].values.astype(float)

        # Baselines
        y_naive = rolling_naive_baseline(y_train_clean.values, y_test)
        y_drift = rolling_drift_baseline(y_train_clean.values, y_test)
        y_holt  = rolling_holt_baseline(y_train_clean.values, y_test)

        # Candidate X
        use_cols = [c for c in x_cols_all if c in cdf.columns]
        X_train_raw = train_df[use_cols].copy()
        X_test_raw  = test_df[use_cols].copy()

        # Keep pandemic dummy even if constant early
        X_train_raw, X_test_raw, kept = prune_features_by_train(
            X_train_raw, X_test_raw, always_keep=DUMMY_FEATS, max_missing=0.35
        )

        dummy_cols = [c for c in DUMMY_FEATS if c in X_train_raw.columns]

        best_mode = None
        best_spec = None
        best_cv = np.inf
        best_feats = []
        best_scaler = None

        # EXOG gating loop
        for mode in EXOG_MODES:
            if mode == "NO_EXOG_ARIMA":
                Xtr_s = None
                spec, cv_rmse = select_best_sarimax_spec(
                    y_train_clean=y_train_clean.values,
                    X_train_scaled=Xtr_s,
                    years_train=train_df["Year"].values
                )
                feats = []
                scaler = None

            elif mode == "DUMMIES_ONLY":
                if len(dummy_cols) == 0:
                    continue
                Xtr = X_train_raw[dummy_cols].copy()
                Xte = X_test_raw[dummy_cols].copy()
                Xtr_imp, Xte_imp = impute_exog_no_leakage(Xtr, Xte)
                Xtr_s, Xte_s, scaler = scale_exog_no_leakage(Xtr_imp, Xte_imp)
                spec, cv_rmse = select_best_sarimax_spec(
                    y_train_clean=y_train_clean.values,
                    X_train_scaled=Xtr_s,
                    years_train=train_df["Year"].values
                )
                feats = list(dummy_cols)

            else:  # TOPK_EXOG
                if len(kept) == 0:
                    continue

                # Impute
                Xtr_imp, Xte_imp = impute_exog_no_leakage(X_train_raw, X_test_raw)

                # Simple top-k by correlation with y (fast + stable)
                def topk_corr(Xtr_imp, y_train_clean, k=5):
                    y = y_train_clean.values.astype(float)
                    scores = {}
                    for col in Xtr_imp.columns:
                        x = Xtr_imp[col].values.astype(float)
                        if np.std(x) < 1e-8 or np.std(y) < 1e-8:
                            scores[col] = 0.0
                        else:
                            scores[col] = abs(np.corrcoef(x, y)[0, 1])
                    top = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:k]
                    return [c for c, _ in top]

                top_cols = topk_corr(Xtr_imp, y_train_clean, k=5)

                # Ensure dummy included
                for dcol in dummy_cols:
                    if dcol not in top_cols:
                        top_cols.append(dcol)

                Xtr = Xtr_imp[top_cols].copy()
                Xte = Xte_imp[top_cols].copy()

                # Scale
                Xtr_s, Xte_s, scaler = scale_exog_no_leakage(Xtr, Xte)

                spec, cv_rmse = select_best_sarimax_spec(
                    y_train_clean=y_train_clean.values,
                    X_train_scaled=Xtr_s,
                    years_train=train_df["Year"].values
                )
                feats = list(top_cols)

            if cv_rmse < best_cv:
                best_cv = cv_rmse
                best_mode = mode
                best_spec = spec
                best_feats = feats
                best_scaler = scaler

        (best_order, best_trend, best_use_log) = best_spec
        print(f"Chosen (by CV): Mode={best_mode} | order={best_order}, trend={best_trend}, log={best_use_log} | CV_RMSE={best_cv:.6f}")

        # Build final exog matrices
        if best_mode == "NO_EXOG_ARIMA":
            X_train_s_final = None
            X_test_s_final = None
            scaler_final = None
        else:
            Xtr = X_train_raw[best_feats].copy()
            Xte = X_test_raw[best_feats].copy()
            Xtr_imp, Xte_imp = impute_exog_no_leakage(Xtr, Xte)
            X_train_s_final, X_test_s_final, scaler_final = scale_exog_no_leakage(Xtr_imp, Xte_imp)

        # SARIMAX forecast
        sarimax_ok = True
        try:
            y_pred_sarimax = expanding_refit_forecast_sarimax(
                y_train_hist=y_train_clean.values,
                X_train_hist=X_train_s_final,
                y_future=y_test,
                X_future=X_test_s_final,
                order=best_order,
                trend=best_trend,
                use_log=best_use_log
            )
        except Exception as e:
            sarimax_ok = False
            y_pred_sarimax = np.full_like(y_test, np.nan, dtype=float)

        # Metrics on test (mask)
        mask = ~np.isnan(y_test)

        sarimax_m = compute_metrics(y_test[mask], y_pred_sarimax[mask]) if sarimax_ok else {"RMSE": np.inf, "MAE": np.inf, "MAPE_%": np.inf, "R2": -np.inf}
        naive_m   = compute_metrics(y_test[mask], y_naive[mask])
        drift_m   = compute_metrics(y_test[mask], y_drift[mask])
        holt_m    = compute_metrics(y_test[mask], y_holt[mask])

        # Best baseline
        baseline_map = {"NAIVE": naive_m["RMSE"], "DRIFT": drift_m["RMSE"], "HOLT": holt_m["RMSE"]}
        best_base_name = min(baseline_map, key=baseline_map.get)
        best_base_rmse = baseline_map[best_base_name]

        # FINAL BEST OF selection
        if sarimax_m["RMSE"] <= best_base_rmse:
            final_model_name = f"SARIMAX({best_mode})"
            y_final = y_pred_sarimax
            final_rmse = sarimax_m["RMSE"]
        else:
            final_model_name = f"BASELINE({best_base_name})"
            y_final = {"NAIVE": y_naive, "DRIFT": y_drift, "HOLT": y_holt}[best_base_name]
            final_rmse = best_base_rmse

        final_m = compute_metrics(y_test[mask], y_final[mask])
        final_mase = mase(y_test[mask], y_final[mask], y_train_clean.values)

        improve_rmse = (best_base_rmse - final_m["RMSE"]) / max(best_base_rmse, 1e-8) * 100.0

        print(
            f"FINAL = {final_model_name} | RMSE={final_m['RMSE']:.6f} | MAE={final_m['MAE']:.6f} | "
            f"MAPE={final_m['MAPE_%']:.2f}% | R2={final_m['R2']:.4f} | MASE={final_mase:.4f} | Improve%={improve_rmse:.2f}"
        )
        print(f"Baselines RMSE: Naive={naive_m['RMSE']:.6f}, Drift={drift_m['RMSE']:.6f}, Holt={holt_m['RMSE']:.6f} | SARIMAX RMSE={sarimax_m['RMSE']:.6f}")

        # Save metrics row
        all_metrics.append({
            "Target": target,
            "Country": country,
            "Final_Model": final_model_name,
            "Final_RMSE": float(final_m["RMSE"]),
            "Final_MAE": float(final_m["MAE"]),
            "Final_MAPE_%": float(final_m["MAPE_%"]),
            "Final_R2": float(final_m["R2"]),
            "Final_MASE": float(final_mase) if not np.isnan(final_mase) else np.nan,
            "BestBaseline": best_base_name,
            "BestBaseline_RMSE": float(best_base_rmse),
            "RMSE_Improvement_%": float(improve_rmse),

            # SARIMAX chosen by CV (for analysis/debug)
            # "Chosen_Exog_Set_CV": best_mode,
            # "Chosen_Exog_Cols_CV": ",".join(best_feats) if best_feats else "",
            # "Best_Order_CV": str(best_order),
            # "Trend_CV": best_trend,
            # "Use_Log_CV": bool(best_use_log),
            # "CV_RMSE": float(best_cv),
            # "SARIMAX_Test_RMSE": float(sarimax_m["RMSE"]) if sarimax_ok else np.inf,
        })

        # Save predictions
        pred_df = pd.DataFrame({
            "Target": target,
            "Country": country,
            "Year": test_df["Year"].values,
            "y_true": y_test,
            "y_final": y_final,
            "final_model": final_model_name,
            "y_sarimax": y_pred_sarimax,
            "y_naive": y_naive,
            "y_drift": y_drift,
            "y_holt": y_holt
        })
        all_preds.append(pred_df)

        # Save SARIMAX artifacts ONLY if SARIMAX is FINAL model
        country_dir = models_dir / country
        country_dir.mkdir(parents=True, exist_ok=True)

        with open(country_dir / "final_choice.txt", "w") as f:
            f.write(f"final_model={final_model_name}\n")
            f.write(f"best_baseline={best_base_name}\n")
            f.write(f"best_baseline_rmse={best_base_rmse}\n")
            f.write(f"final_rmse={final_m['RMSE']}\n")

        if final_model_name.startswith("SARIMAX"):
            # save scaler + features + spec
            if scaler_final is not None:
                dump(scaler_final, country_dir / "scaler.joblib")
            with open(country_dir / "features.txt", "w") as f:
                f.write("\n".join(best_feats))
            with open(country_dir / "best_spec.txt", "w") as f:
                f.write(f"mode={best_mode}\n")
                f.write(f"order={best_order}\n")
                f.write(f"trend={best_trend}\n")
                f.write(f"use_log={best_use_log}\n")
                f.write(f"cv_rmse={best_cv}\n")

            # Fit on full data up to 2022
            full_df = cdf[(cdf["Year"] >= TRAIN_START) & (cdf["Year"] <= TEST_END)].copy()
            y_full = full_df[target].astype(float).interpolate(limit_direction="both").ffill().bfill().clip(lower=0)

            if best_mode == "NO_EXOG_ARIMA":
                X_full_s = None
            else:
                X_full = full_df[best_feats].copy()
                # impute using train-style medians
                Xtr_tmp = X_train_raw[best_feats].copy().interpolate(limit_direction="forward").ffill()
                med = Xtr_tmp.median(numeric_only=True)
                X_full = X_full.interpolate(limit_direction="forward").ffill().fillna(med)
                X_full_s = scaler_final.transform(X_full.values)

            y_full_t = transform_y(y_full.values, best_use_log)
            final_model = SARIMAX(
                y_full_t,
                exog=X_full_s,
                order=best_order,
                trend=best_trend,
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            final_res = final_model.fit(disp=False)
            final_res.save(country_dir / "final_sarimax.pkl")

    metrics_df = pd.DataFrame(all_metrics).sort_values(["Target", "Country"])
    preds_df = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()

    target_dir.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(target_dir / "metrics_by_country.csv", index=False)
    preds_df.to_csv(target_dir / "predictions_test_2017_2022_bestof.csv", index=False)

    if not metrics_df.empty:
        overall = metrics_df[["Final_MAE", "Final_RMSE", "Final_MAPE_%", "Final_R2", "Final_MASE", "BestBaseline_RMSE", "RMSE_Improvement_%"]].mean().to_frame().T
        overall.insert(0, "Target", target)
        overall.to_csv(target_dir / "metrics_overall_avg.csv", index=False)

    return metrics_df, preds_df



# 8) RUN

edu_metrics_v2, edu_preds_v2 = train_evaluate_education_bestof(TARGET, X_COLS_ALL)

display(edu_metrics_v2)
display(edu_preds_v2.head())

print("\nOverall averages (Education) - FINAL BEST-OF:")
summary = edu_metrics_v2.groupby("Target")[["Final_MAE", "Final_RMSE", "Final_MAPE_%", "Final_R2", "Final_MASE", "BestBaseline_RMSE", "RMSE_Improvement_%"]].mean().round(6)
display(summary)


Dropping all-NaN columns: ['Conflict_Intensity']
Shape: (155, 24)
Countries: ['CHN' 'FRA' 'GBR' 'RUS' 'USA']
Year range: 1994 - 2024
Candidate Education X columns: ['D_Expenditure_GDP', 'GDP_Growth_Annual', 'Inflation_Annual', 'Unemployment_Total_of_TLF', 'Population', 'EduExp_GovShare_IMF_Pct', 'Secondary_SclEnroll_Gross_Pct', 'D_Expenditure_GDP_lag1', 'GDP_Growth_Annual_lag1', 'Inflation_Annual_lag1', 'Unemployment_Total_of_TLF_lag1', 'Population_lag1', 'EduExp_GovShare_IMF_Pct_lag1', 'Secondary_SclEnroll_Gross_Pct_lag1', 'D_PANDEMIC']

Country: USA
Chosen (by CV): Mode=TOPK_EXOG | order=(0, 1, 0), trend=n, log=False | CV_RMSE=0.334068
FINAL = BASELINE(NAIVE) | RMSE=0.235504 | MAE=0.178492 | MAPE=3.44% | R2=-0.1769 | MASE=1.3088 | Improve%=0.00
Baselines RMSE: Naive=0.235504, Drift=0.260993, Holt=0.300032 | SARIMAX RMSE=0.236268

Country: RUS
Chosen (by CV): Mode=TOPK_EXOG | order=(1, 0, 1), trend=n, log=True | CV_RMSE=0.160658
FINAL = BASELINE(NAIVE) | RMSE=0.568098 | MAE=0.387567 |

,Target,Country,Final_Model,Final_RMSE,Final_MAE,Final_MAPE_%,Final_R2,Final_MASE,BestBaseline,BestBaseline_RMSE,RMSE_Improvement_%
2,Education_Expenditure_GDP,CHN,SARIMAX(NO_EXOG_ARIMA),0.198385,0.162067,4.050357,-0.390111,1.068531,NAIVE,0.198385,0.0
4,Education_Expenditure_GDP,FRA,BASELINE(HOLT),0.137291,0.098503,1.784034,-0.665081,1.295365,HOLT,0.137291,0.0
3,Education_Expenditure_GDP,GBR,SARIMAX(NO_EXOG_ARIMA),0.463942,0.327177,6.261826,-1.381725,2.198648,NAIVE,0.463942,0.0
1,Education_Expenditure_GDP,RUS,BASELINE(NAIVE),0.568098,0.387567,9.440966,-1.363311,2.452654,NAIVE,0.568098,0.0
0,Education_Expenditure_GDP,USA,BASELINE(NAIVE),0.235504,0.178492,3.444524,-0.176890,1.308803,NAIVE,0.235504,0.0


,Target,Country,Year,y_true,y_final,final_model,y_sarimax,y_naive,y_drift,y_holt
0,Education_Expenditure_GDP,USA,2017,5.09297,4.78328,BASELINE(NAIVE),4.705101,4.78328,4.724081,4.571307
1,Education_Expenditure_GDP,USA,2018,4.89502,5.09297,BASELINE(NAIVE),5.031252,5.09297,5.049810,5.054836
2,Education_Expenditure_GDP,USA,2019,4.95747,4.89502,BASELINE(NAIVE),4.859119,4.89502,4.845410,4.849780
3,Education_Expenditure_GDP,USA,2020,5.39532,4.95747,BASELINE(NAIVE),5.453423,4.95747,4.912343,4.917596
4,Education_Expenditure_GDP,USA,2021,5.42038,5.39532,BASELINE(NAIVE),5.227888,5.39532,5.368769,5.371729



Overall averages (Education) - FINAL BEST-OF:


,Final_MAE,Final_RMSE,Final_MAPE_%,Final_R2,Final_MASE,BestBaseline_RMSE,RMSE_Improvement_%
Target,,,,,,,
Education_Expenditure_GDP,0.230761,0.320644,4.996341,-0.795424,1.6648,0.320644,0.0
